In [1]:
### load packages
import math
import glob

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns
plt.rcParams.update({'font.size': 14})

In [4]:
# load propagate results
propagate_lst = []
for file in glob.glob('../figure_6/cf_metag_results/2026-04-08_outputs/assemblyanalyze/propagate/*/*.propagate.tsv.gz'):
    # skip if empty df
    try:
        df = (
            pl.read_csv(file, separator='\t', null_values=['NA'], columns=['prophage', 'prophage_len', 'host', 'CohenD', 'host_len', 'prophage-host_ratio'])
                .with_columns([
                    pl.col('prophage').cast(pl.String),
                    pl.col('host').cast(pl.String),
                    pl.col('prophage_len').cast(pl.Int64),
                    pl.col('CohenD').cast(pl.Float64),
                    pl.col('prophage-host_ratio').cast(pl.Float64),
                    pl.col('prophage').str.split('_k').list[0].alias('sample_id')
                ])
        )
        propagate_lst.append(df)
    except:
        pass

propagate_df = (
    pl.concat(propagate_lst)
)
propagate_df.write_csv('cf_metag_propagate.tsv', separator='\t')

In [5]:
# identify mvirs active viruses in unenriched
!zcat ../figure_6/cf_metag_results/2026-04-08_outputs/assemblyanalyze/mvirs/*/*.mvirs.fasta.gz | \
    grep "^>" > cf_metag_mvirs_contig_ids.txt

mvirs_contigs = (
    pl.read_csv('cf_metag_mvirs_contig_ids.txt', separator=' ', has_header=False, new_columns=['mvirs_id'])
        .with_columns([
            pl.col('mvirs_id').str.replace('>', '').str.split(':').list[0].alias('contig_id_no_prophage'),
            pl.col('mvirs_id').str.split('OPRs=').list[1].str.split('-').list[0].cast(pl.Int64).alias('num_oprs'),
            pl.col('mvirs_id').str.split('HSs=').list[1].str.split('-').list[0].cast(pl.Int64).alias('num_clipped'),
        ])
        .with_columns([
            pl.col('contig_id_no_prophage').str.split('_k').list[0].alias('sample_id')
        ])
)
mvirs_contigs.write_csv('cf_metag_mvirs.tsv', separator='\t')

In [6]:
### load coverm results to identify viruses detected in bulk and enriched
df_lst = []

for file in glob.glob('../figure_6/cf_metag_results/2026-04-08_outputs/referenceanalyze/coverm/*/*.coverm.tsv.gz'):
    sample_id = file.split('/')[-1].split('.')[0]
    group = sample_id.rsplit('_', 1)[0]
    try:
        df = (
            pl.read_csv(file, separator='\t', new_columns=['contig_id', 'trimmed_mean', 'mean', 'variance', 'covered_bases', 'length'])
                .with_columns([
                    pl.lit(sample_id).alias('sample_id'),
                    pl.lit(group).alias('group')
                ])
                .with_columns([
                    (pl.col('covered_bases')/pl.col('length')).alias('breadth'),
                ])
                .with_columns([
                    (1 - math.e**(-0.833 * pl.col('mean'))).alias('expected_breadth'),
                ])
                .with_columns([
                    (pl.col('breadth')/pl.col('expected_breadth')).alias('breadth_ratio'),
                ])
        )
        df_lst.append(df)
    except:
        continue


coverm_df = pl.concat(df_lst)
coverm_df.write_csv('cf_metag_coverm_results.tsv', separator='\t')

In [7]:
phage_host_ratio_lst = []
sylph_tax_lst = []

for file in glob.glob('../figure_6/cf_metag_results/2026-04-08_outputs/referenceanalyze/sylphtax/*/*.sylph-tax.tsv.gz'):
    sample_id=file.split('/')[-1].split('.syl')[0]
    df = (
        pl.read_csv(
            file, separator='\t', skip_rows=1, new_columns=['clade_name', 'taxonomic_abundance', 'relative_abundance', 'ani', 'coverage', 'virus_host'], null_values=['NA'],
            schema_overrides={'clade_name': pl.Utf8, 'taxonomic_abundance': pl.Float64, 'relative_abundance': pl.Float64, 'ani': pl.Float64, 'coverage': pl.Float64, 'virus_host': pl.Utf8}
        )
        .with_columns([
            pl.lit(sample_id).alias('sample_id'),
        ])
    )
    virus_df = df.filter(pl.col('virus_host').is_not_null()).with_columns([pl.col('virus_host').str.replace_all(';', '|')])
    sylph_tax_lst.append(df)
    bac_df = df.filter(pl.col('clade_name').str.starts_with('d__Bacteria'))
    phage_host_ratio_lst.append(
        virus_df
            .join(bac_df, left_on='virus_host', right_on='clade_name', how='inner')
            .with_columns([
                (pl.col('taxonomic_abundance') / pl.col('taxonomic_abundance_right')).alias('phage_host_ratio'),
                pl.col('clade_name').str.split('t__').list[1].alias('votu_rep'),
                pl.col('clade_name').str.replace(r'.*vSPECIES-', '').str.replace(r'\|.*', '').cast(pl.Int64).alias('species_cluster_id'),
            ])
    )

sylph_tax_df = pl.concat(sylph_tax_lst)
phage_host_ratio_df = pl.concat(phage_host_ratio_lst)[['sample_id', 'species_cluster_id', 'phage_host_ratio']]
sylph_tax_df.write_csv('cf_metag_sylph_tax_results.tsv', separator='\t')

In [9]:
for file in glob.glob('../figure_6/cf_metag_results/2026-04-08_outputs/preprocess/*/.command.log'):
    # get sample name
    sample_name = file.split('/')[-2]
    bioproject = file.split('/')[-5]
    # parse log to count number of bp after deacon
    with open(file, 'r') as log:
        for line in log:
            if line.startswith('Retained '):
                num_bp = int(line.split()[4].split('/')[0])

    with open('cf_metag_read_depth.txt', 'a') as f_out:
        f_out.write(f'{sample_name}\t{num_bp}\n')